In [ ]:
# In this notebook, we will see what will happen if we have 30% of data per enginge
# Importing libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pathlib as pl
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from xgboost import XGBRegressor
from pathlib import Path
# Informative sensors
informative_sensors = [
    'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
    'sensor_9', 'sensor_11', 'sensor_12', 'sensor_14',
    'sensor_15', 'sensor_20', 'sensor_21'
]
# Column names
column_names = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
# Load the dataset
Data_raw = Path('..')/ 'data'/ 'raw'
train_df = pd.read_csv(Data_raw / 'train_FD001.txt', sep='\s+', header=None, names=column_names)
test_df = pd.read_csv(Data_raw / 'test_FD001.txt', sep='\s+', header=None, names=column_names)
rul_df = pd.read_csv(Data_raw / 'RUL_FD001.txt', sep='\s+', header=None, names=['RUL'])
# RUL construction
train_df['RUL'] = train_df.groupby('engine_id')['cycle'].transform(max) - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper=125)
# Train-test split
train_engines = train_df['engine_id'].unique()
split_idx = int(len(train_engines) * 0.8)
train = train_df[train_df['engine_id'].isin(train_engines[:split_idx])]
val = train_df[train_df['engine_id'].isin(train_engines[split_idx:])]
# Normalization
scaler = StandardScaler()
train_scaled = train.copy()
train_scaled[informative_sensors] = scaler.fit_transform(train[informative_sensors])
val_scaled = val.copy()
val_scaled[informative_sensors] = scaler.transform(val[informative_sensors])
# Limited data per engine
def limited_data(df, fraction=0.3):
    limited =[]
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        n_cycles = int(len(engine_data) * fraction)
        limited.append(engine_data.head(n_cycles))
    return pd.concat(limited, ignore_index=True)
train_limited = limited_data(train_scaled, fraction=0.3)
# Gaussian noise augmentation
def augment_data(df, n_copies=3, noise_level=0.01):
    augmented = [df]
    for _ in range(n_copies):
        noisy = df.copy()
        noisy[informative_sensors]= df[informative_sensors] + np.random.normal(0, noise_level, size=df[informative_sensors].shape)
        augmented.append(noisy)
    return pd.concat(augmented, ignore_index=True)
# Sequence creation for LSTM
def create_sequences(df, sequence_length=30):
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_data = df[df['engine_id'] == engine_id]
        for i in range(len(engine_data) - sequence_length):
            X.append(engine_data[informative_sensors].iloc[i:i+sequence_length].values)
            y.append(engine_data['RUL'].iloc[i+sequence_length])
    return np.array(X), np.array(y)
# Augmenting limited data
train_augmented = augment_data(train_limited, n_copies=3, noise_level=0.01)
X_train_aug, y_train_aug = create_sequences(train_augmented)
X_val_seq, y_val_seq = create_sequences(val_scaled)

# LSTM
model = Sequential([
    LSTM(64, input_shape=(30, 11)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
# Early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights = True)

history = model.fit(X_train_aug, y_train_aug,
                    validation_data=(X_val_seq, y_val_seq),
                    epochs=50, batch_size=64
                    , callbacks=[early_stopping])

y_pred = model.predict(X_val_seq).flatten()
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_val_seq, y_pred)
r2 = r2_score(y_val_seq, y_pred)
print(f'Limited LSTM RMSE: {rmse:.2f}, R2: {r2:.2f}')
